# CelebA-HQ DDPM training

Clones the repo from GitHub, uses a local CelebA-HQ dataset uploaded into the Colab session, and trains with the same pipeline while saving checkpoints to Google Drive.


## Mount Drive and clone repo

Update `REPO_URL` if needed.


In [ ]:
import os
import shutil
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

REPO_URL = "https://github.com/AAnanya19/Human-Faces-Generation-Diffusion-Models.git"
BRANCH = "main"
WORKDIR = Path("/content/Human-Faces-Generation-Diffusion-Models")

DRIVE_ROOT = Path("/content/drive/MyDrive/aml")
RUNS_ROOT = DRIVE_ROOT / "ddpm_runs"
RUN_NAME = "celebahq_run_001"
RUN_DIR = RUNS_ROOT / RUN_NAME

if WORKDIR.exists():
    shutil.rmtree(WORKDIR)

!git clone --branch "$BRANCH" "$REPO_URL" "$WORKDIR"

assert WORKDIR.exists(), f"Repo folder not found: {WORKDIR}"
RUN_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", WORKDIR)
print("Run dir:", RUN_DIR)


## Install dependencies


In [ ]:
!pip install -q datasets huggingface_hub tqdm matplotlib torchvision


## Prepare local dataset

upload either:
- a folder named `celeba_hq_256` into the Colab Files pane, or
- a zip file named `celeba_hq_256.zip`

This notebook trains from the local `/content` copy for speed and avoids saving the dataset into Drive.


In [ ]:
from pathlib import Path

LOCAL_FOLDER = Path("/content/celeba_hq_256")
LOCAL_ZIP = Path("/content/celeba_hq_256.zip")
print("Expected local folder:", LOCAL_FOLDER)
print("Expected local zip:", LOCAL_ZIP)


## Verify local CelebA-HQ dataset

If `celeba_hq_256` is already extracted in `/content`, it is reused. Otherwise, if `celeba_hq_256.zip` is present, it is extracted once and then verified.


In [ ]:
import shutil
import zipfile
from pathlib import Path

DATASET_DIR = Path("/content/celeba_hq_256")
EXPECTED_IMAGE_COUNT = 30000
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp"}

def count_images(root: Path) -> int:
    return sum(1 for p in root.rglob("*") if p.suffix.lower() in IMAGE_EXTENSIONS)

image_count = count_images(DATASET_DIR) if DATASET_DIR.exists() else 0

if image_count >= EXPECTED_IMAGE_COUNT:
    print("Using existing local dataset:", DATASET_DIR)
else:
    if not LOCAL_ZIP.exists() and not DATASET_DIR.exists():
        raise FileNotFoundError(
            "Upload `celeba_hq_256` or `celeba_hq_256.zip` into the Colab session Files pane first."
        )
    if LOCAL_ZIP.exists():
        print("Extracting uploaded zip:", LOCAL_ZIP)
        if DATASET_DIR.exists():
            shutil.rmtree(DATASET_DIR)
        with zipfile.ZipFile(LOCAL_ZIP, "r") as zf:
            zf.extractall(Path("/content"))
    image_count = count_images(DATASET_DIR)

print("Dataset directory:", DATASET_DIR)
print("Image files found:", image_count)
if image_count < EXPECTED_IMAGE_COUNT:
    raise RuntimeError(
        f"Local dataset looks incomplete: found {image_count}, expected about {EXPECTED_IMAGE_COUNT}."
    )


## Training config


In [ ]:
EPOCHS = 300
BATCH_SIZE = 16
IMAGE_SIZE = 64
LR = 1e-4
TIMESTEPS = 1000
WEIGHT_DECAY = 1e-4
GRAD_CLIP = 1.0
SAMPLE_EVERY = 10
NUM_SAMPLE_IMAGES = 8
CHECKPOINT_EVERY = 25
FOLDER_SUBSET_SIZE = 3000
FOLDER_TEST_SIZE = 300
BASE_CHANNELS = 64
TIME_DIM = 256
CHANNEL_MULTS = "1,2,4,8"
NUM_RES_BLOCKS = 2
DROPOUT = 0.1
ATTENTION_RESOLUTIONS = "16,8"
FIXED_SAMPLE_SEED = 123
FIXED_TRAJECTORY_SEED = 321
TRAJECTORY_SAVE_EVERY = 100


## Run training


In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch:", torch.__version__)
print("device:", device)
if device != "cuda":
    raise RuntimeError("GPU is not active. In Colab, switch Runtime -> Change runtime type -> GPU.")

!python3 /content/Human-Faces-Generation-Diffusion-Models/src/training/train.py \
    --epochs $EPOCHS \
    --batch_size $BATCH_SIZE \
    --image_size $IMAGE_SIZE \
    --lr $LR \
    --timesteps $TIMESTEPS \
    --weight_decay $WEIGHT_DECAY \
    --grad_clip $GRAD_CLIP \
    --sample_every $SAMPLE_EVERY \
    --num_sample_images $NUM_SAMPLE_IMAGES \
    --checkpoint_every $CHECKPOINT_EVERY \
    --dataset_source folder \
    --dataset_path "$DATASET_DIR" \
    --folder_subset_size $FOLDER_SUBSET_SIZE \
    --folder_test_size $FOLDER_TEST_SIZE \
    --base_channels $BASE_CHANNELS \
    --time_dim $TIME_DIM \
    --channel_mults $CHANNEL_MULTS \
    --num_res_blocks $NUM_RES_BLOCKS \
    --dropout $DROPOUT \
    --attention_resolutions $ATTENTION_RESOLUTIONS \
    --fixed_sample_seed $FIXED_SAMPLE_SEED \
    --fixed_trajectory_seed $FIXED_TRAJECTORY_SEED \
    --trajectory_save_every $TRAJECTORY_SAVE_EVERY \
    --device cuda \
    --save_dir "$RUN_DIR"


## Inspect saved outputs


In [ ]:
print("Checkpoint files:", sorted(str(p.name) for p in RUN_DIR.glob("*.pth")))
print("Sample grids:", sorted(str(p.name) for p in (RUN_DIR / "generated_samples").glob("*.png")))
print("Trajectory grids:", sorted(str(p.name) for p in (RUN_DIR / "trajectories").glob("*.png")))
print("Loss log exists:", (RUN_DIR / "loss_log.csv").is_file())
print("Metadata file exists:", (RUN_DIR / "run_config.json").is_file())
print((RUN_DIR / "run_config.json").read_text())
